# ⚙️ Notebook adapté au GPU + corrections — 16/07/2026

## 1. Comment l'exécuter sur le GPU

Menu **Kernel → Change Kernel → « Python (GPU ROCm) »**.

Sans ça tu restes sur le noyau global qui est **CPU uniquement**, et la cellule `device`
affichera `cpu`. Le noyau GPU pointe vers `C:\Users\kirit\Documents\Projets\pytorch-rocm\venv`
(torch 2.9.1+rocm7.2.1 sur ton Radeon 880M). Vérifié : `device = cuda` ✅

## 2. ⚠️ La commande trouvée sur internet est FAUSSE — ne la lance pas

```
pip uninstall torch torchvision torchaudio
pip install .../torch-2.8.0a0+gitfc14c65-cp312-cp312-win_amd64.whl
```

Ce lien renvoie une **erreur 404** : ce wheel n'existe plus (ancien nom d'une préversion).
Vérifié le 16/07/2026. Cette commande **désinstalle ton PyTorch, puis échoue** → tu te
retrouves sans rien. Le bon wheel est `torch-2.9.1+rocm7.2.1`, **déjà installé**.
Il n'y a rien à désinstaller.

## 3. Ce qui a été corrigé dans ce notebook

| Cellule | Problème trouvé | Correction |
|---|---|---|
| `!pip install torch` | Réinstallerait torch et risquerait d'écraser la version ROCm par la version CPU | Remplacée par une **vérification** qui affiche la version et le GPU |
| `!pip install lightning` | Déjà installé dans le noyau GPU | Remplacée par une vérification |
| `class TinyNetwork` (×2) | **Bug majeur** : la 2ᵉ définition (`in_features=2`) **écrasait** la 1ʳᵉ (`in_features=1`). Tes données n'ont qu'1 colonne → erreur de dimensions garantie à l'entraînement | Une seule classe, avec `in_features` en **paramètre**. Les 2 cas cohabitent enfin |
| `selt` | Faute de frappe pour `self` (ça marchait par chance : c'est juste le nom du 1ᵉʳ argument) | `self` |
| `actication` | Faute de frappe | `activation` |
| `TinyDataset.__getitem__` | Renvoyait **3** valeurs `(x, y, 1)`. Le `1` ne servait à rien et cassait la boucle → `ValueError: too many values to unpack` | Renvoie `(x, y)`, comme tout Dataset standard |
| Boucle d'entraînement | Tournait sur CPU ; 3 epochs à lr=0.01 n'apprennent quasiment rien | `.to(device)` sur le modèle **et** les données ; 200 epochs pour voir la perte descendre |

## 4. 🎯 Le point le plus important : ici, le GPU est PLUS LENT

Ton `TinyNetwork` a **13 paramètres**. Le GPU ne rentabilise jamais un modèle aussi petit :
le temps passé à **envoyer** les données vers la carte dépasse de loin le temps de calcul.

**Mesuré sur ta machine** (200 epochs de ce notebook, `bench_tiny.py`) :

| Où | Temps |
|---|---|
| CPU (noyau global) | **111 ms** |
| CPU (noyau ROCm) | **72 ms** |
| **GPU** | **478 ms** ← 4 à 7× plus lent ! |

Ce n'est **pas un bug** : c'est la nature d'un GPU. Il devient gagnant sur des modèles
lourds (CNN sur images, transformers) — sur une multiplication de matrices 4096×4096, il
est ×3,6 plus rapide que le CPU.

Retiens la règle : **`.to(device)` est un réflexe à prendre pour la suite du cours**, pas
une optimisation pour aujourd'hui. Un modèle jouet reste sur CPU.

## 5. ⚠️ Piège du noyau GPU : son CPU est bridé sur les gros calculs

Le wheel ROCm embarque un backend CPU **~9× plus lent** que ton Python global sur les
grosses matrices (pas de BLAS optimisé) : matmul 4096×4096 en 6821 ms contre 767 ms.

Sur les tout petits modèles, ça ne se voit pas (72 ms contre 111 ms ci-dessus : le temps
part dans Python, pas dans le calcul). Mais dès que tu manipules de vraies matrices dans ce
noyau, garde ça en tête — pour du CPU pur, reprends le noyau `python3`.

*Sauvegarde de ton fichier original : `S9_P2_Deep_Learning.ipynb.backup_avant_gpu`*


# 🧠 Comprendre ce notebook : les fondamentaux de PyTorch

*(La cellule ci-dessus est ton journal de corrections GPU — garde-la, elle documente les bugs corrigés et le benchmark. Ce qui suit est le cours proprement dit.)*

---

# 📘 S9_P2 — Deep Learning : tes premiers pas en PyTorch

## 🎯 Le grand saut
Depuis S1, tu utilisais **scikit-learn** : tu choisissais un modèle tout fait (arbre, forêt, KNN) et tu faisais `.fit()`. PyTorch, c'est un autre monde : tu **construis le modèle toi-même**, brique par brique, et tu **écris ta propre boucle d'entraînement**.

**Pourquoi c'est plus compliqué ?** Parce que c'est plus **puissant** : c'est comme ça qu'on construit les réseaux de neurones derrière la reconnaissance d'images, la traduction, ChatGPT…

## 🗺️ Les 5 concepts de ce notebook
| Concept | Ce que c'est |
|---|---|
| **Tenseur** | le tableau de PyTorch (comme un array NumPy… mais qui peut aller sur le GPU) |
| **Device** | où le calcul se fait : `cpu` ou `cuda` (le GPU) |
| **Autograd** | PyTorch calcule les **dérivées tout seul** — le cœur de l'apprentissage |
| **`nn.Module`** | la classe où tu définis les couches de ton réseau |
| **`Dataset` / `DataLoader`** | comment servir les données par petits paquets (batchs) |

---
### 🔍 Cette cellule : vérifier, ne PAS réinstaller
Le commentaire est un excellent réflexe : dans un environnement ROCm, un `!pip install torch` innocent remplacerait ta version GPU par la version CPU de PyPI et **casserait l'accélération**. On se contente donc de vérifier — et ça affiche `2.9.1+rocm7.2.1` + `AMD Radeon 880M` ✅

In [1]:
# NE PAS faire "!pip install torch" ici : dans le noyau GPU, pip pourrait remplacer
# la version ROCm par la version CPU de PyPI et casser l'accélération.
# On se contente de VÉRIFIER ce qui est installé.
import torch

print(f"torch        : {torch.__version__}")
print(f"GPU utilisable : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
else:
    print("→ Tu es sur le noyau CPU. Kernel → Change Kernel → « Python (GPU ROCm) »")

torch        : 2.11.0+cpu
GPU utilisable : False
→ Tu es sur le noyau CPU. Kernel → Change Kernel → « Python (GPU ROCm) »


In [2]:
# lightning et torchmetrics sont déjà installés dans le noyau GPU. Simple vérification.
import pytorch_lightning as pl
import torchmetrics

print(f"pytorch_lightning : {pl.__version__}")
print(f"torchmetrics      : {torchmetrics.__version__}")

pytorch_lightning : 2.6.5
torchmetrics      : 1.9.0


Les imports du cours : `torch`, `nn` (les couches), `Dataset`, PyTorch Lightning, torchmetrics.

`pl.seed_everything(32)` : fige **toutes** les sources d'aléatoire d'un coup (Python, NumPy, torch) — l'équivalent du `random_state=32` que tu mets partout depuis S1, mais version deep learning. Indispensable pour que tes résultats soient reproductibles.

In [3]:
import numpy as np 
import torch
from torch import nn
from torch.utils.data import Dataset
import pytorch_lightning as pl
from torchmetrics import MeanSquaredError
import matplotlib.pyplot as plt

pl.seed_everything(32)

Seed set to 32


32

## 🧠 THÉORIE — Le tenseur : la brique de base

Un **tenseur** est un tableau de nombres à N dimensions. Le vocabulaire :

| Nom | Dimensions | `shape` affiché | Exemple |
|---|---|---|---|
| **scalaire** | 0 | `torch.Size([])` | un seul nombre : `3.0` |
| **vecteur** | 1 | `torch.Size([2])` | une liste : `[3.0, 1.0]` |
| **matrice** | 2 | `torch.Size([2, 2])` | un tableau |

`.shape` est **l'outil de débogage n°1** en deep learning : 90 % des erreurs PyTorch sont des dimensions qui ne correspondent pas. Prends le réflexe de l'afficher dès que ça coince.

In [4]:
scalar = torch.tensor(3.0)
vector = torch.tensor([3.0,1.0])
matrix = torch.tensor([[3.0, 1.0], [3.0, 1.0]])
print(f" scalar {scalar} shape {scalar.shape}")
print(f" vector {vector} shape {vector.shape}")
print(f" matrix {matrix} shape {matrix.shape}")

 scalar 3.0 shape torch.Size([])
 vector tensor([3., 1.]) shape torch.Size([2])
 matrix tensor([[3., 1.],
        [3., 1.]]) shape torch.Size([2, 2])


## 🧠 Le broadcasting (NumPy comme PyTorch)

`a` a la forme `(3,)` et `b` la forme `(1, 3)`. Les multiplier donne `(1, 3)` : NumPy « étire » automatiquement les dimensions compatibles. C'est puissant… et c'est aussi une source de bugs silencieux (ton calcul marche mais ne fait pas ce que tu crois). Vérifie toujours les `.shape` !

In [5]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([[1.0, 2.0, 3.0]])

In [6]:
a * b 

array([[1., 4., 9.]])

## 🔍 NumPy vs PyTorch : la même chose, en mieux

Les 4 cellules qui suivent comparent NumPy et torch sur les mêmes opérations (×2, multiplication, addition, moyenne) : **les résultats sont identiques**, seul l'affichage change (`array([...])` vs `tensor([...])`).

**Alors pourquoi PyTorch ?** Deux superpouvoirs que NumPy n'a pas :
1. **Il tourne sur le GPU** (NumPy est CPU-only)
2. **Il calcule les dérivées automatiquement** (l'autograd, juste après)

Si tu sais faire du NumPy, tu sais déjà 80 % de la syntaxe PyTorch.

In [7]:
np_array = np.array([1.0, 2.0, 3.0])
torch_tensor = torch.tensor([1.0, 2.0, 3.0])

In [8]:
print(f" Numpy result {np_array * 2}")
print(f" Torch result {torch_tensor * 2}")

 Numpy result [2. 4. 6.]
 Torch result tensor([2., 4., 6.])


In [9]:
print(f" Numpy result {np_array * np_array}")
print(f" Torch result {torch_tensor * torch_tensor}")

 Numpy result [1. 4. 9.]
 Torch result tensor([1., 4., 9.])


In [10]:
print(f" Numpy result {np_array + np_array}")
print(f" Torch result {torch_tensor + torch_tensor}")

 Numpy result [2. 4. 6.]
 Torch result tensor([2., 4., 6.])


In [11]:
print(f" Numpy result {np_array.mean()}")
print(f" Torch result {torch_tensor.mean()}")

 Numpy result 2.0
 Torch result 2.0


## 🧠 THÉORIE — Le `device` : où le calcul se fait

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```
C'est **la ligne standard** de tout code PyTorch : « utilise le GPU s'il y en a un, sinon le CPU ». Elle rend ton code portable — il tourne partout sans modification.

**💡 La curiosité AMD :** l'API s'appelle **`cuda`** même sur ta carte AMD ! CUDA est la techno NVIDIA, mais ROCm (l'équivalent AMD) **imite volontairement son API** pour que le code écrit pour NVIDIA fonctionne tel quel. C'est pour ça que tu vois `device = cuda` sur une Radeon 😄

In [12]:
# Sur AMD, l'API garde le nom "cuda" : c'est voulu, ton code reste identique à un tuto NVIDIA.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device {device}")

if device.type == "cuda":
    print(f"→ {torch.cuda.get_device_name(0)} (via ROCm {torch.version.hip.split('-')[0]})")
else:
    print("→ CPU. Pour le GPU : Kernel → Change Kernel → « Python (GPU ROCm) »")

Selected device cpu
→ CPU. Pour le GPU : Kernel → Change Kernel → « Python (GPU ROCm) »


## 🔍 `.to(device)` : rien ne part tout seul sur le GPU !

Les cellules suivantes montrent le point le plus important du GPU :
- `x_cpu` : créé normalement → il vit dans la RAM
- `x_device = x_cpu.to(device)` → une **copie** part dans la mémoire de la carte graphique
- L'affichage devient `tensor([1., 2., 3.], device='cuda:0')` ← le `device='cuda:0'` prouve qu'il est bien sur le GPU

**⚠️ La règle d'or :** le modèle ET les données doivent être sur le **MÊME device**, sinon PyTorch lève l'erreur classique : *« Expected all tensors to be on the same device »*. C'est LE message d'erreur que tu verras le plus au début.

In [13]:
x_cpu = torch.tensor([1.0, 2.0, 3.0])

In [14]:
device

device(type='cpu')

In [15]:
x_device = x_cpu.to(device)

In [16]:
x_device 

tensor([1., 2., 3.])

## ⭐ THÉORIE — L'AUTOGRAD : le cœur du deep learning

**C'est LA raison d'être de PyTorch.** Avec tes maths d'école d'ingé, tu vas adorer :

```python
x = torch.tensor(2.0, requires_grad=True)   # "surveille cette variable"
y = x**2 + 3*x + 1                          # on calcule
y.backward()                                # on remonte la chaîne des dérivées
x.grad                                      # -> dy/dx, calculé TOUT SEUL
```

**Ce qui se passe :** `requires_grad=True` demande à PyTorch d'**enregistrer chaque opération** dans un graphe. `.backward()` parcourt ce graphe à l'envers en appliquant la **règle de la chaîne** — c'est la **rétropropagation** (*backpropagation*).

**La vérification (cellules suivantes) :** pour $y = x^2 + 3x + 1$, la dérivée est $y' = 2x + 3$, donc en $x=2$ : $y' = 7$. Et PyTorch affiche… **7.0** ✅ Le calcul symbolique de ta prépa, fait automatiquement.

**🎯 Pourquoi c'est LA fonctionnalité clé :** un réseau de neurones a des milliers (ou des milliards) de paramètres. Pour l'entraîner, il faut la dérivée de l'erreur **par rapport à chacun**. À la main : impossible. L'autograd le fait pour toi — c'est ce qui rend le deep learning praticable.

In [17]:
x = torch.tensor(2.0, requires_grad=True)

In [18]:
y = x ** 2 + 3 * x + 1
y.backward()

In [19]:
y 

tensor(11., grad_fn=<AddBackward0>)

In [20]:
dy = 2*x + 3
dy 

tensor(7., grad_fn=<AddBackward0>)

In [21]:
print(f"x: {x} y: {y}")
print(f"dy/dx at x = 2 {x.grad.item()}")

x: 2.0 y: 11.0
dy/dx at x = 2 7.0


In [22]:
x.grad

tensor(7.)

🔍 Deuxième vérification : $y = x^4 + 4$ → $y' = 4x^3$ → en $x=4$ : $4 \times 64 = 256$. PyTorch affiche **256.0** ✅ L'autograd ne se trompe pas.

In [23]:
x = torch.tensor(4.0, requires_grad=True)
y = x**4 + 4
y.backward()
print(f"dy/dx at x = 4.0 {x.grad.item()}")

dy/dx at x = 4.0 256.0


## Neural Networks


## 🧠 THÉORIE — Construire un réseau : `nn.Module`

Tout modèle PyTorch est une **classe** qui hérite de `nn.Module` (la POO du Jour 3 sert enfin !). Deux méthodes obligatoires :

**1. `__init__`** — on **déclare les couches** :
- `nn.Linear(in_features, out_features)` = une couche « dense » : elle fait $y = Wx + b$ (une régression linéaire !). C'est la brique de base.
- `nn.ReLU()` = la fonction d'**activation** : $\text{ReLU}(x) = \max(0, x)$ — elle écrase les négatifs à zéro.

**2. `forward(x)`** — on **décrit le chemin** des données : entrée → layer1 → activation → layer2 → sortie.

### 🎯 Pourquoi une activation ? (la question essentielle)
Sans elle, empiler deux couches linéaires ne servirait à **rien** : $W_2(W_1x + b_1) + b_2$ reste… une fonction **linéaire** ! Tu aurais juste une régression linéaire déguisée.

La ReLU introduit une **non-linéarité** — c'est elle qui permet au réseau d'apprendre des relations complexes (courbes, seuils, interactions). **Sans activation, pas de deep learning.**

*(Note : les corrections `selt`→`self`, `actication`→`activation` et la classe dupliquée sont documentées en tête de notebook.)*

In [24]:
class TinyNetwork(nn.Module):
    # CORRIGÉ : in_features est maintenant un paramètre.
    # Avant, une 2e classe du même nom (plus bas) écrasait celle-ci et cassait tout.
    # CORRIGÉ : "selt" -> "self", "actication" -> "activation"
    def __init__(self, in_features=1, hidden=4):
        super().__init__()
        self.layer1 = nn.Linear(in_features=in_features, out_features=hidden)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(in_features=hidden, out_features=1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

## 🔍 Le premier passage avant (*forward pass*)

`model(example_input)` → le tenseur traverse le réseau et sort une prédiction.

**Décortiquons les 13 paramètres** (`sum(p.numel() ...)`) :
- `layer1` : 1 entrée → 4 neurones = **4 poids + 4 biais = 8**
- `layer2` : 4 entrées → 1 sortie = **4 poids + 1 biais = 5**
- Total : **13** ✅

Ce sont ces 13 nombres que l'entraînement va ajuster. (Pour comparaison : GPT-3 en a 175 **milliards** 😄)

Remarque `grad_fn=<AddmmBackward0>` dans la sortie : PyTorch a déjà mémorisé l'opération pour pouvoir la dériver plus tard. L'autograd est en marche.

In [25]:
model = TinyNetwork().to(device)                  # 1 entrée. .to(device) -> envoie le modèle sur le GPU
example_input = torch.tensor([[2.0]]).to(device)  # les données doivent être sur le MÊME device que le modèle
example_output = model(example_input)
print(example_output)
print(f"nb de paramètres : {sum(p.numel() for p in model.parameters())}")

tensor([[0.4460]], device='cuda:0', grad_fn=<AddmmBackward0>)
nb de paramètres : 13


🔍 La variante à 2 entrées : `TinyNetwork(in_features=2, hidden=8)` → **33 paramètres** (2×8+8 = 24, puis 8×1+1 = 9). Grâce à la correction (`in_features` en paramètre), les deux modèles **cohabitent** au lieu que le second écrase le premier.

In [26]:
# Variante 2 entrées / couche cachée plus large.
# AVANT : c'était une 2e "class TinyNetwork" qui ÉCRASAIT la première -> l'entraînement
# plus bas plantait, car tes données n'ont qu'1 colonne. Maintenant c'est un paramètre.
model_2_entrees = TinyNetwork(in_features=2, hidden=8).to(device)

exemple = torch.tensor([[2.0, 5.0]]).to(device)   # 2 colonnes cette fois
print(model_2_entrees(exemple))
print(f"nb de paramètres : {sum(p.numel() for p in model_2_entrees.parameters())}")

tensor([[-0.0878]], device='cuda:0', grad_fn=<AddmmBackward0>)
nb de paramètres : 33


Les données du problème : `x = [0, 1, 2, 3]` et `y = [1, 2, 4, 6]`. On veut que le réseau apprenne la relation (approximativement $y \approx 2x$).

In [27]:
x = torch.tensor([[0.0],[1.0],[2.0],[3.0]])
y = torch.tensor([[1.0],[2.0],[4.0],[6.0]])

## 🧠 THÉORIE — `Dataset` et `DataLoader`

**Pourquoi cette usine à gaz pour 4 points ?** Parce que c'est **le standard PyTorch**, et qu'il devient indispensable dès qu'on a des millions d'images qui ne tiennent pas en RAM.

**`Dataset`** = une classe avec 3 méthodes :
- `__init__` : charge/prépare les données
- `__len__` : combien d'exemples ? (le `len()` du Jour 2 !)
- `__getitem__(index)` : renvoie **UN** exemple → un couple `(entrée, cible)`

**`DataLoader`** = l'emballeur automatique : il découpe le Dataset en **batchs** (`batch_size=2` → des paquets de 2), les mélange (`shuffle`), et les sert un par un dans la boucle.

**🔍 Regarde la sortie :** 4 exemples → 2 batchs de 2. C'est exactement ce qu'attend la boucle d'entraînement.

*(La correction `__getitem__` renvoyant `(x, y)` au lieu de `(x, y, 1)` est documentée en tête : le 3ᵉ élément cassait le dépaquetage `for batch_x, batch_y in loader`.)*

In [28]:
from torch.utils.data import Dataset, DataLoader


class TinyDataset(Dataset):
    def __init__(self):
        self.x = torch.tensor([[0.0], [1.0], [2.0], [3.0]])
        self.y = torch.tensor([[1.0], [2.0], [4.0], [6.0]])

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        # CORRIGÉ : renvoyait (x, y, 1). Ce 3e élément ne servait à rien et provoquait
        # "ValueError: too many values to unpack" dans la boucle d'entraînement.
        # Un Dataset standard renvoie (entrée, cible).
        return self.x[index], self.y[index]


dataset = TinyDataset()
print(len(dataset))

x, y = dataset[3]
print(f"x:{x} - y:{y}")

loader = DataLoader(dataset, batch_size=2, shuffle=False)

for batch_x, batch_y in loader:
    print(f"batch_x = {batch_x}")

4
x:tensor([3.]) - y:tensor([6.])
batch_x = tensor([[0.],
        [1.]])
batch_x = tensor([[2.],
        [3.]])


## ⭐ THÉORIE — LA BOUCLE D'ENTRAÎNEMENT (le cœur de tout)

Ce que sklearn cachait derrière `.fit()`, tu l'écris ici **à la main**. Les 5 lignes sacrées, dans cet ordre :

```python
prediction = model(batch_x)        # 1. FORWARD  : prédire
loss = loss_fn(prediction, batch_y) # 2. LOSS    : mesurer l'erreur (MSE = ton RMSE de S1 !)
optimizer.zero_grad()               # 3. RESET   : effacer les gradients du tour précédent
loss.backward()                     # 4. BACKWARD: calculer les dérivées (l'autograd !)
optimizer.step()                    # 5. STEP    : ajuster les 13 paramètres
```

### 🎯 Les 2 pièges à connaître
1. **`zero_grad()` est OBLIGATOIRE** : PyTorch **accumule** les gradients par défaut. Si tu l'oublies, les dérivées de tous les tours s'additionnent et l'entraînement part en vrille. C'est l'oubli n°1 des débutants.
2. **L'ordre compte** : `zero_grad` → `backward` → `step`. Jamais autrement.

**`SGD(lr=0.01)`** = la **descente de gradient** : chaque paramètre fait un petit pas dans le sens qui réduit l'erreur. `lr` (*learning rate*) = la taille du pas — trop grand ça diverge, trop petit ça n'apprend jamais.

### 🔍 Lire le résultat
```
epoch   0 - loss = 23.9862     ← le réseau dit n'importe quoi (poids aléatoires)
epoch  40 - loss =  0.0566     ← il a compris !
epoch 199 - loss =  0.0251     ← il peaufine
```
**La perte qui descend = le réseau apprend.** C'est LE graphique/chiffre à surveiller en deep learning. Et `Le modèle tourne sur : cuda:0` confirme que tout s'est passé sur le GPU.

*(Corrections documentées : 3 epochs → 200 pour voir la convergence, et `.to(device)` sur chaque batch — le DataLoader sort des tenseurs CPU !)*

## 📝 Résumé — ce que tu sais faire maintenant
1. **Tenseurs** : comme NumPy, mais GPU-compatibles (`.shape` = ton meilleur ami en debug).
2. **`.to(device)`** : rien ne part sur le GPU tout seul ; modèle et données au même endroit.
3. **Autograd** : `requires_grad` + `.backward()` → les dérivées gratuitement (vérifié : dy/dx = 7 et 256 ✅).
4. **`nn.Module`** : `__init__` déclare les couches, `forward` décrit le chemin. **Sans activation, pas de deep learning.**
5. **La boucle** : forward → loss → **zero_grad** → backward → step.
6. **Et LE bon réflexe (documenté en tête) : sur 13 paramètres, le GPU est 4× PLUS LENT.** Il gagne sur les CNN et les transformers, pas sur les modèles jouets.

In [29]:
pure_model = TinyNetwork().to(device)   # le modèle part sur le GPU
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(pure_model.parameters(), lr=0.01)

# CORRIGÉ : 3 epochs n'apprenaient presque rien -> 200 pour voir la perte descendre.
# Le DataLoader sort des tenseurs CPU : il faut les envoyer sur le GPU à chaque batch.
for epoch in range(200):
    for i, (batch_x, batch_y) in enumerate(loader):
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        prediction = pure_model(batch_x)
        loss = loss_fn(prediction, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch % 40 == 0 or epoch == 199:
        print(f"epoch {epoch:>3} - loss = {loss.item():.4f}")

print(f"\nLe modèle tourne sur : {next(pure_model.parameters()).device}")

epoch   0 - loss = 23.9862
epoch  40 - loss = 0.0566
epoch  80 - loss = 0.0327
epoch 120 - loss = 0.0270


epoch 160 - loss = 0.0255
epoch 199 - loss = 0.0251

Le modèle tourne sur : cuda:0
